# GOAT-Evolution — End-to-end Colab training

Runs the **full** GOAT-Evolution per-attack pipeline inside Colab:

1. Clone the repo and install the `backend` package.
2. Configure `HF_TOKEN` + `GEMINI_API_KEY`.
3. Evaluate the baseline LLM defender on a fixed holdout suite.
4. Run real episodes with **Gemini-driven** Red/Blue agents + **Gemini judge** + **LoRA fine-tune per attack** on a 2B/3B Hugging Face model.
5. Plot the learning curves (violation, refusal-quality, ASR) and a baseline-vs-trained bar chart.
6. Push the trained adapter + per-attack JSONL shards to Hugging Face.

OpenEnv-compliant: every episode goes through `GoatEnvironment.reset / step / state` and the LLM defender is the test subject.


## 1. Install dependencies

In [ ]:
%pip install -q -U \
    "transformers>=4.44" \
    "trl>=0.13" \
    "peft>=0.12" \
    "datasets>=2.20" \
    "accelerate>=0.34" \
    "huggingface_hub>=0.26" \
    "openenv-core>=0.2.1" \
    "google-generativeai>=0.8.3" \
    "matplotlib>=3.8" \
    "pydantic>=2.9" \
    "fastapi>=0.115" \
    "sse-starlette>=2.1" \
    "httpx>=0.27"

## 2. Clone the repo and add `backend` to the Python path

Set `REPO_URL` to your fork (or HF Space) and rerun if needed.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL  = os.environ.get('GOAT_REPO_URL',  'https://github.com/YOUR_USER/goat-evolution.git')
REPO_DIR  = pathlib.Path('/content/goat-evolution')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=False)

PROJECT = REPO_DIR / 'goat_evolution' if (REPO_DIR / 'goat_evolution').exists() else REPO_DIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print('Working dir:', os.getcwd())

## 3. Secrets

* `HF_TOKEN` for pushing the trained adapter and shards.
* `GEMINI_API_KEY` to drive the Red/Blue agents and the judge.

On Colab, store these in the Secrets panel (🔑 icon).

In [ ]:
import os
try:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'GEMINI_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = input('Paste HF_TOKEN (required for hub push/login): ').strip()
from huggingface_hub import login
login(os.environ['HF_TOKEN'])
if not os.environ.get('GEMINI_API_KEY'):
    print('GEMINI_API_KEY not set -> notebook will default to scripted agents.')

## 4. Pick the test subject (2B or 3B)

In [ ]:
SUBJECT_SIZE    = '3b'             # '0.5b' | '1.5b' | '2b' | '3b'
SUBJECT_BACKEND = 'hf_local'       # 'stub' (no GPU) or 'hf_local' (real LoRA)
EPISODES        = 3                # quick smoke default; bump for stronger curves
TASK_ID         = 4
MODEL_PROFILE   = 'standard_model'
ARTIFACTS_DIR   = '/content/artifacts'
LOGS_DIR        = '/content/logs'
RUN_LABEL       = 'colab_full'
RED_PROVIDER    = 'scripted'       # switch to 'gemini' after setting GEMINI_API_KEY
BLUE_PROVIDER   = 'scripted'       # switch to 'gemini' after setting GEMINI_API_KEY
JUDGE_PROVIDER  = 'scripted'       # switch to 'gemini' after setting GEMINI_API_KEY
HF_DATASET_REPO = ''               # e.g. 'YOUR_USER/goat-evolution-attacks' (optional push)
HF_MODEL_REPO   = ''               # e.g. 'YOUR_USER/goat-evolution-3b-defender' (optional push)
PUSH_EACH_ATTACK = bool(HF_DATASET_REPO or HF_MODEL_REPO)
print({
    'SUBJECT_SIZE': SUBJECT_SIZE,
    'SUBJECT_BACKEND': SUBJECT_BACKEND,
    'EPISODES': EPISODES,
    'providers': (RED_PROVIDER, BLUE_PROVIDER, JUDGE_PROVIDER),
})

## 5. Baseline holdout (untrained defender)

In [ ]:
from backend.training.gating import HoldoutGate
from backend.training.llm_subject import LLMSubject
from backend.cli.realtime_trainer import SUBJECT_MODEL_ALIASES

subject_baseline = LLMSubject(backend=SUBJECT_BACKEND, model_name=SUBJECT_MODEL_ALIASES[SUBJECT_SIZE])
baseline_metrics = HoldoutGate().evaluate(subject_baseline)
print('Baseline:', baseline_metrics.to_dict())

## 6. Run the per-attack training pipeline (Red·Blue·Judge = Gemini, Subject = HF LoRA)

In [ ]:
import asyncio
from pathlib import Path
from backend.cli.realtime_trainer import RealtimeTrainer
from backend.training.llm_factory import build_agents
from backend.training.logging_setup import setup_logging

setup_logging(logs_root=LOGS_DIR, run_label=RUN_LABEL)

if (RED_PROVIDER == 'gemini' or BLUE_PROVIDER == 'gemini' or JUDGE_PROVIDER == 'gemini') and not os.environ.get('GEMINI_API_KEY'):
    print('GEMINI_API_KEY missing; forcing scripted providers for this run.')
    RED_PROVIDER = BLUE_PROVIDER = JUDGE_PROVIDER = 'scripted'

agents = build_agents(
    red_provider_name=RED_PROVIDER,
    blue_provider_name=BLUE_PROVIDER,
    judge_provider_name=JUDGE_PROVIDER,
)
trainer = RealtimeTrainer(
    artifacts_root=Path(ARTIFACTS_DIR),
    subject_size=SUBJECT_SIZE,
    backend=SUBJECT_BACKEND,
    push_each_attack=PUSH_EACH_ATTACK,
    hf_dataset_repo=HF_DATASET_REPO or None,
    hf_model_repo=HF_MODEL_REPO or None,
    gate_min_violation_drop=0.0,
    gate_max_refusal_drop=0.05,
    learning_rate=5e-5,
    agents=agents,
)
print('[trainer] subject:', trainer.subject_size, '/', trainer.subject_model_name, 'backend=', trainer.subject.backend)
print('[agents] red=', RED_PROVIDER, 'blue=', BLUE_PROVIDER, 'judge=', JUDGE_PROVIDER)

async def _run_episodes():
    for i in range(1, EPISODES + 1):
        report = await trainer.run_episode(task_id=TASK_ID, model_profile=MODEL_PROFILE)
        print(f'[ep {i}/{EPISODES}] turns={report.turns} breaking={report.breaking_attacks} '
              f'promotions={report.promotions} rollbacks={report.rollbacks} '
              f'red={report.cumulative_red_reward:.2f} blue={report.cumulative_blue_reward:.2f} '
              f'serving_v={report.serving_version_after}')
asyncio.run(_run_episodes())

## 7. Generate evidence artifacts (reward curves + baseline vs trained bars)

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('bvt', PROJECT / 'training' / 'baseline_vs_trained.py')
bvt = importlib.util.module_from_spec(spec); spec.loader.exec_module(bvt)
sys.argv = ['bvt', '--artifacts-root', ARTIFACTS_DIR]
bvt.main()
from IPython.display import Image, display, JSON
import json, pathlib
ev = pathlib.Path(ARTIFACTS_DIR) / 'evidence'
display(JSON(json.loads((ev / 'summary.json').read_text())))
for png in ('reward_curve.png', 'holdout_compare.png'):
    p = ev / png
    if p.exists():
        display(Image(filename=str(p)))

## 8. Inspect the per-component logs

In [ ]:
import pathlib
log_dir = pathlib.Path(LOGS_DIR) / RUN_LABEL
for path in sorted(log_dir.glob('*.log')):
    print(f'--- {path.name} ({path.stat().st_size} bytes) ---')
    print(path.read_text().splitlines()[-3:])

## 9. (Optional) Push the trained adapter to Hugging Face

If you set `HF_MODEL_REPO` above and `--push-each-attack` was true, every promoted candidate was already uploaded under `checkpoints/<label>/`. To force-push the *current* serving checkpoint as well:

In [ ]:
if HF_MODEL_REPO:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True, repo_type='model')
    serving_label = trainer.state_store.state.serving_label
    serving_dir = pathlib.Path(ARTIFACTS_DIR) / 'checkpoints' / serving_label
    if serving_dir.exists():
        api.upload_folder(folder_path=str(serving_dir), path_in_repo=f'checkpoints/{serving_label}', repo_id=HF_MODEL_REPO, repo_type='model', commit_message=f'Promote {serving_label}')
        print('Pushed', serving_label, 'to', HF_MODEL_REPO)